# Lab 1.2: Conversational RAG - Mortgage Policy Assistant

In this lab, we will build a **Banking Policy Assistant** for Wells Fargo. This assistant allows loan officers to query internal mortgage guidelines (e.g., Conforming vs. Jumbo loans) and ask follow-up questions while maintaining conversational context.

## Use Case
A loan officer needs to quickly check credit score requirements for different loan types without searching through a 200-page PDF manual. They might ask:
1. "What is the min credit score for a Jumbo loan?"
2. "How does that compare to FHA?" (Contextual follow-up)

## Key Concepts
1. **History Aware Retriever**: Rephrases the user's latest query using the chat history so it makes sense as a standalone query to the vector store.
2. **Chat History**: Maintaining line of conversation.

In [21]:
# 1. Install Dependencies
print("Installing dependencies...")
%pip install -qU langchain langchain-groq langchain-community langchain-huggingface chromadb sentence-transformers
print("Dependencies installed.")

Installing dependencies...
Dependencies installed.


In [22]:
# 2. Setup API Keys
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
# Optional for LangSmith Tracking
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_TRACING_V2"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "RAG Conversational"

In [23]:
fruits_details = """
Organic Fruits retailers

1- Tata fruits : This contains fruits of Tata made with fresh ingredients. Its premium quality with 0% preservatives .
2- Apple fruits : This contains fruits of Apple made with fresh ingredients. qulaity is 96%
"""

In [24]:
# 3. Setup Vector Store (Mortgage Policy Data)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Load Internal Policy Document
print("Loading fruits retailers details...")
docs = [Document(page_content=fruits_details, metadata={"source": "Organic Fruits retailers"})]
print(f"Loaded {len(docs)} document(s).")

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(docs)
print(f"Split into {len(splits)} chunks.")

# Embed & Store
print("Creating vector store...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()
print("Vector store created.")

Loading fruits retailers details...
Loaded 1 document(s).
Split into 1 chunks.
Creating vector store...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store created.


## 4. History Aware Retriever
We need a chain that takes the `chat_history` and the `input` and generates a search query.

In [25]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Initialize LLM
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed"
)

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# Chain to rephrase question
contextualize_q_chain = contextualize_q_prompt | llm | StrOutputParser()
print("Contextualization chain created.")

Contextualization chain created.


In [26]:
# Let's test the contextualization to understand what it does
from langchain_core.messages import HumanMessage, AIMessage

print("--- Testing Query Contextualization ---")
# Mock history: user asked about Jumbo loans, AI answered.
sample_history = [
    HumanMessage(content="What is apple fruit quality?"),
    AIMessage(content="")
]
# User asks a follow-up specific to the context (referring to 'that')
sample_input = "what is company name?"
print(f"Chat History: {len(sample_history)} messages")
print(f"User Follow-up: {sample_input}")

rephrased_query = contextualize_q_chain.invoke({"chat_history": sample_history, "input": sample_input})
print(f"Rephrased Query (for Vector Store): {rephrased_query}")
print("---------------------------------------")

--- Testing Query Contextualization ---
Chat History: 2 messages
User Follow-up: what is company name?
Rephrased Query (for Vector Store): what is company name?
---------------------------------------


## 5. QA Chain with History
Now we create the final chain that uses the retrieved documents to answer.

In [27]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_system_prompt = (
    "You are a specialized Mortgage Policy Assistant for Wells Fargo. "
    "Use the following pieces of retrieved policy context to answer "
    "the loan officer's question. If you don't know the answer, say that you "
    "cannot find it in the policy. Keep answers professional and concise."
    "\n\n"
    "{context}"
)

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

def contextualized_question(input: dict):
    if input.get("chat_history"):
        return contextualize_q_chain
    else:
        return input.get("input")

rag_chain = (
    RunnablePassthrough.assign(
        context=contextualized_question | retriever | format_docs
    )
    | qa_prompt
    | llm
    | StrOutputParser()
)
print("Full RAG chain created.")

Full RAG chain created.


## 6. Testing the Chat
We can now manage specific chat sessions.

In [29]:
chat_history = []

# First Question: Specific Policy Query
user_input = "What is apple fruit quality?"

print(f"User: {user_input}")
# rag_chain returns string now
answer = rag_chain.invoke({"input": user_input, "chat_history": chat_history})
print(f"AI: {answer}")

# Update History
chat_history.extend([HumanMessage(content=user_input), AIMessage(content=answer)])

# Second Question (Follow-up): Contextual Comparison
print("\n--- Follow up ---")
user_input = "What is tata fruits made of?"
print(f"User: {user_input}")

answer = rag_chain.invoke({"input": user_input, "chat_history": chat_history})
print(f"AI: {answer}")

chat_history.extend([HumanMessage(content=user_input), AIMessage(content=answer)])

User: What is apple fruit quality?
AI: Based on the provided information, **Apple fruits** are described as having a quality of **96%**. The policy context does not specify additional details about quality beyond this percentage.

--- Follow up ---
User: What is tata fruits made of?
AI: Based on the provided information, **Tata fruits** are made with **fresh ingredients** and are described as having **0% preservatives**. The policy context does not specify further details about the exact composition of the ingredients.
